# Monte Carlo Simulation Analysis

In [1]:
import numpy as np
import pandas as pd
import awkward as ak
import uproot

In [2]:
import matplotlib.pyplot as plt
from IPython.display import Image, display
from matplotlib.colors import LogNorm
# Aesthetics:
fs = 14    # fontsize

In [3]:
# Check out the structure
path = "/home/pira/Documenti/PoD/LCP/LCP_B/ALICE/AO2DtreeMC.root"
file = uproot.open(path)
file.classnames()

{'DF_2303121152302944;1': 'TDirectory',
 'DF_2303121152302944/O2mccollision;1': 'TTree',
 'DF_2303121152302944/O2collision_001;1': 'TTree',
 'DF_2303121152302944/O2filtertrack;1': 'TTree',
 'DF_2303121152302944/O2filtertrackextr;1': 'TTree',
 'DF_2303121152302944/O2filtertrackmc;1': 'TTree',
 'DF_2303121152302944/O2genparticles;1': 'TTree',
 'DF_2303121152302976;1': 'TDirectory',
 'DF_2303121152302976/O2mccollision;1': 'TTree',
 'DF_2303121152302976/O2collision_001;1': 'TTree',
 'DF_2303121152302976/O2filtertrack;1': 'TTree',
 'DF_2303121152302976/O2filtertrackextr;1': 'TTree',
 'DF_2303121152302976/O2filtertrackmc;1': 'TTree',
 'DF_2303121152302976/O2genparticles;1': 'TTree',
 'DF_2303121152303008;1': 'TDirectory',
 'DF_2303121152303008/O2mccollision;1': 'TTree',
 'DF_2303121152303008/O2collision_001;1': 'TTree',
 'DF_2303121152303008/O2filtertrack;1': 'TTree',
 'DF_2303121152303008/O2filtertrackextr;1': 'TTree',
 'DF_2303121152303008/O2filtertrackmc;1': 'TTree',
 'DF_2303121152303008

In [4]:
file["DF_2303121152302944/O2filtertrackmc"].show()

name                 | typename                 | interpretation                
---------------------+--------------------------+-------------------------------
fPdgCode             | int32_t                  | AsDtype('>i4')
fIsPhysicalPrimary   | bool                     | AsDtype('bool')
fMainHfMotherPdgCode | int32_t                  | AsDtype('>i4')
fMainBeautyAncest... | int32_t                  | AsDtype('>i4')
fMainMotherOrigIndex | int32_t                  | AsDtype('>i4')
fMainMotherNfinal... | int32_t                  | AsDtype('>i4')
fMainMotherPt        | float                    | AsDtype('>f4')
fMainMotherY         | float                    | AsDtype('>f4')
fMainBeautyAncest... | float                    | AsDtype('>f4')
fMainBeautyAncestorY | float                    | AsDtype('>f4')


## Useful variable for the MC analysis
- ### O2filtertrackmc
   One entry per reconstructed track containing simulation information of the particle associated to the reconstructed track
  - **fPdgCode**: dentifies particle species according to convention reported in PDG; e.g. π +(-) : (-)211; K +(-) :(-)321; p (p) : (-)2212;
  - **fMainHfMotherPdgCode**: pdg code of the mother when relevant, 0 otherwise
    - HF particles we want to study: $D^0 (D^0): (-)421; \ Λ_c^+ (Λ_c^-): (-)4122$;
    - $K_0^s: \ 310$;
  - **fMainMotherOrigIndex**: index of mother particle in original simulation tree.
    In simulation many particles (quarks, gluons, hadrons) are produced. Some particles may even appear
    more than once (e.g. think to a quark radiated a gluon, or to a particle which interacts with the material).
    Book keeping them all, makes the tree size very large. I saved you only the index and a filtered tree
    (O2genparticles), which contains the information of only the particles we are interested in;
  - **fMainMotherNfinalStateDaught**: number of final-state daughters, negative in
    case the final state does not match one in which we are interested (e.g. $D_0 → K^- K^+ , \ Λ_c^+ → p(K_0^s →)π^- π^+ $)
    Usage: it’s not enough that two or three particles come from the same $D_0$ or $Λ_c^+$ to identify signal, you must be sure that these $D_0$ and $Λ_c^+$ decayed in the right channel;
  - **fMainMotherPt**, **fMainMotherY**: $p_T$ and rapidity $y$ of mother particle;
  - **fMainBeautyAncestorPdgCode**: pdg code of beauty particles for cases in which the charm hadrons derive from beauty decay, 0 otherwise;
  - **fMainBeautyAncestorPt**, **fMainBeautyAncestorY**: $p_T$ and rapidity $y$ of beauty ancestor particle;

In summary: a pair of tracks corresponds to a $D_0 → K^-π^+$ decay if:
- fPdgCode are -321 and 211
- fMainHfMotherPdgCode = 421 for both
- fMainMotherOrigIndex is the same for the two tracks
- fMainMotherNfinalStateDaught = 2 (for both)

- ### O2genparticles
  Table of filtered generated particles.
As previously mentioned, in simulation many particles (quarks, gluons, hadrons) are produced.
Some particles may even appear more than once (e.g. think to a quark radiated a gluon, or to a
particle which interacts with the material). Book keeping them all, makes the tree size very large. I saved you only a filtered tree, which contains the information of only the particles we are interested in: $D_0$ and $Λ_c^+$ (and $K_s^0$)
    - **fPdgCode**: pdg code of generated particle (same convention as in O2filtertrackmc);
    - **fMainMotherPt**, **fMainMotherY**: particle pT and rapidity. Sorry for possibly misleading name: “mother” here it is used in correspondence to “main mother” in O2filtertrackmc;
    - **fMaxEtaDaughter**: max abs(pseudorapidity, η) of daughter particles. Needed to
identify reconstructable decays, for which daughters must have |η|<0.8;
    - **fMainBeautyAncestor**: [PdgCode,Pt,Y]: same as in O2filtertrackmc;
    - **fIndexMcCollisions**: match to generated collision index of tree entry in O2mccollision table → needed only to select collision with |fPosz| <10 cm;


- ### O2mccollision
   Table of generated collisions
    - **fPosX, fPosY, fPosZ**: global coordinates of collision point. (n.b.: primary vertex in O2collision_001 table gives coordinates of primary vertex, i.e. of the reconstructed position of what is assumed to be a collision)
You will need only the fPosZ variable to reject collisions generated with at
|fPosZ| <10 cm
Apply same condition in reconstruction: i.e. reject tracks from collision whose
primary vertex has |fPosZ| <10 cm

In [5]:
# visualize the tables to work with
file["DF_2303121152302944/O2filtertrackmc"].arrays(library="pd", entry_stop=15)

,fPdgCode,fIsPhysicalPrimary,fMainHfMotherPdgCode,fMainBeautyAncestorPdgCode,fMainMotherOrigIndex,fMainMotherNfinalStateDaught,fMainMotherPt,fMainMotherY,fMainBeautyAncestorPt,fMainBeautyAncestorY
0,211,True,0,0,-1,0,0.0,0.0,0.0,0.0
1,-211,True,0,0,-1,0,0.0,0.0,0.0,0.0
2,-211,True,0,0,-1,0,0.0,0.0,0.0,0.0
3,211,True,0,0,-1,0,0.0,0.0,0.0,0.0
4,211,True,0,0,-1,0,0.0,0.0,0.0,0.0
5,-211,True,0,0,-1,0,0.0,0.0,0.0,0.0
6,211,True,0,0,-1,0,0.0,0.0,0.0,0.0
7,-211,True,0,0,-1,0,0.0,0.0,0.0,0.0
8,211,True,0,0,-1,0,0.0,0.0,0.0,0.0
9,211,True,0,0,-1,0,0.0,0.0,0.0,0.0


In [6]:
file["DF_2303121152302944/O2filtertrack"].arrays(library="pd", entry_stop=15)

,fIndexCollisions,fIsInsideBeamPipe,fTrackType,fX,fAlpha,fY,fZ,fSnp,fTgl,fSigned1Pt
0,0,1,1,0.016007,2.475195,0.033457,4.265919,1.010531e-07,-0.372177,0.752879
1,0,1,1,-0.007946,1.927238,0.038384,4.259802,-2.123390e-08,0.038139,-1.221110
2,0,1,1,-0.043304,0.307420,-0.012610,4.266381,3.352523e-08,0.475846,-0.868761
3,0,1,1,-0.031706,1.312997,0.035398,4.264633,7.280882e-08,-0.700898,1.675801
4,0,1,1,0.039679,-2.137382,-0.011104,-8.138128,-5.869162e-08,0.204762,1.419873
5,0,1,1,0.020985,-1.526217,-0.045100,4.265929,-3.990283e-08,0.800033,-0.808950
6,0,1,1,-0.018552,-0.604253,-0.040316,4.268981,6.518571e-08,-0.088289,2.135623
7,0,1,1,-0.041649,0.893735,0.022228,4.254848,1.120295e-08,-0.414245,-3.106939
8,0,1,1,-0.029618,-0.305636,-0.040014,4.264381,9.391308e-09,-0.634567,2.418539
9,0,1,1,0.042234,-2.923403,0.014537,4.262048,-1.808300e-10,0.078375,3.215032


In [7]:
z_cut_test = file["DF_2303121152302944/O2filtertrack"].arrays(library="pd")
z_cut_test = z_cut_test[z_cut_test["fZ"].abs() < 10]
row_indexis = z_cut_test.index.tolist()
print(len(z_cut_test), max(row_indexis))
z_cut_test.head(10)


5838 6190


,fIndexCollisions,fIsInsideBeamPipe,fTrackType,fX,fAlpha,fY,fZ,fSnp,fTgl,fSigned1Pt
0,0,1,1,0.016007,2.475195,0.033457,4.265919,1.010531e-07,-0.372177,0.752879
1,0,1,1,-0.007946,1.927238,0.038384,4.259802,-2.123390e-08,0.038139,-1.221110
2,0,1,1,-0.043304,0.307420,-0.012610,4.266381,3.352523e-08,0.475846,-0.868761
3,0,1,1,-0.031706,1.312997,0.035398,4.264633,7.280882e-08,-0.700898,1.675801
4,0,1,1,0.039679,-2.137382,-0.011104,-8.138128,-5.869162e-08,0.204762,1.419873
5,0,1,1,0.020985,-1.526217,-0.045100,4.265929,-3.990283e-08,0.800033,-0.808950
6,0,1,1,-0.018552,-0.604253,-0.040316,4.268981,6.518571e-08,-0.088289,2.135623
7,0,1,1,-0.041649,0.893735,0.022228,4.254848,1.120295e-08,-0.414245,-3.106939
8,0,1,1,-0.029618,-0.305636,-0.040014,4.264381,9.391308e-09,-0.634567,2.418539
9,0,1,1,0.042234,-2.923403,0.014537,4.262048,-1.808300e-10,0.078375,3.215032


In [8]:
# Filter the data to select the "good tracks"
# Perform the cut on fPosZ in the "track" file and then perform a "JOIN"
tracks_df = file["DF_2303121152302944/O2filtertrackmc"].arrays(library="pd")
tracks_df = tracks_df.loc[row_indexis]

# Select the right particles (fPdgCode)
filtered_tracks = tracks_df[tracks_df["fPdgCode"].isin([-321,211])]

# Select all the rows with fMainHfMotherPdgCode = 421 AND fMainMotherNfinalStateDaught = 2
filtered_tracks = filtered_tracks[(filtered_tracks["fMainHfMotherPdgCode"]==421) 
                    & (filtered_tracks['fMainMotherNfinalStateDaught'] == 2)]

# fMainMotherOrigIndex is the same for the grouped tracks
filtered_tracks = filtered_tracks.groupby('fMainMotherOrigIndex')

# Function to filter out the "good" couples of tracks
def check_group(group):
    part_pdg = set(group['fPdgCode'])
    return -321 in part_pdg and 211 in part_pdg and len(group) == 2 

# Applica il filtro sui gruppi
D_0_decays = filtered_tracks.filter(check_group)

print("In this file there are", len(D_0_decays)/2, "D_0 decays")
D_0_decays.head(10)



In this file there are 18.0 D_0 decays


,fPdgCode,fIsPhysicalPrimary,fMainHfMotherPdgCode,fMainBeautyAncestorPdgCode,fMainMotherOrigIndex,fMainMotherNfinalStateDaught,fMainMotherPt,fMainMotherY,fMainBeautyAncestorPt,fMainBeautyAncestorY
535,211,True,421,0,115539,2,4.748023,0.561990,0.000000,0.000000
536,-321,True,421,0,115539,2,4.748023,0.561990,0.000000,0.000000
1278,-321,True,421,0,278327,2,2.138527,-0.015906,0.000000,0.000000
1284,211,True,421,0,278327,2,2.138527,-0.015906,0.000000,0.000000
2060,-321,True,421,-521,457113,2,10.691305,0.422080,14.177983,0.358982
2084,211,True,421,-521,457113,2,10.691305,0.422080,14.177983,0.358982
2264,-321,True,421,-521,489049,2,1.257220,-0.045749,1.209267,-0.296353
2268,211,True,421,-521,489049,2,1.257220,-0.045749,1.209267,-0.296353
2316,-321,True,421,511,502279,2,3.601864,0.189009,10.104934,0.209708
2317,211,True,421,511,502279,2,3.601864,0.189009,10.104934,0.209708


In [ ]:
# Do the same of the previous cell but with all the data (not only one file)
# Prepare the data
file_names_mc = file.keys(filter_name=r"*O2filtertrackmc")
file_names = file.keys(filter_name=r"*O2filtertrack")
all_tracks_mc = pd.concat( [file[name_string].arrays(library="pd") for name_string in file_names_mc] )
all_tracks = pd.concat( [file[name_string].arrays(library="pd") for name_string in file_names] )

# Select the rows to mantain
z_cut = all_tracks[all_tracks["fZ"].abs() < 10]
row_indexis = z_cut.index.tolist() 

# Perform the cut on the z coordinate
all_tracks_mc = all_tracks_mc.loc[row_indexis]

# Filter the data selecting the right particles that mark the D_0 decay
filtered_tracks = all_tracks_mc[all_tracks_df["fPdgCode"].isin([-321,211])]

# Select all the rows with fMainHfMotherPdgCode = 421 AND fMainMotherNfinalStateDaught = 2
filtered_tracks = filtered_tracks[(filtered_tracks["fMainHfMotherPdgCode"]==421) 
                    & (filtered_tracks['fMainMotherNfinalStateDaught'] == 2)]

# fMainMotherOrigIndex is the same for the grouped tracks
filtered_tracks = filtered_tracks.groupby('fMainMotherOrigIndex')

# Function to filter out the "good" couples of tracks
def check_group(group):
    part_pdg = set(group['fPdgCode'])
    return -321 in part_pdg and 211 in part_pdg and len(group) == 2 

# Applica il filtro sui gruppi
D_0_decays = filtered_tracks.filter(check_group)

print("In the full dataset there are", len(D_0_decays)/2, "D_0 decays")
#D_0_decays.head(10)

In [ ]:
# Now we have to check if these tracks come from collisions with |fPosZ| < 10 cm
# Start checking this in the O2mccollision tables
# Select the collisions with fPosZ < 10 cm
#file_names_collisions = file.keys(filter_name=r"*O2mccollision")
#mccollision_df = pd.concat( [ file[name].arrays(library="pd") for name in file_names_collisions ] )

mccollision_df = file["DF_2303121152302944/O2mccollision"].arrays(library="pd")
mccollision_pZ_cut = mccollision_df[mccollision_df["fPosZ"].abs() < 10]

# visualize the result
print(len(mccollision_pZ_cut), len(mccollision_df))
mccollision_pZ_cut.head(10)


In [ ]:
# Import all the data in a single dataframe
#file_names_gen_par = file.keys(filter_name=r"*O2genparticles")
#generated_part = pd.concat( [ file[name].arrays(library="pd") for name in file_names_gen_par ] )

# Match the ID of the two tables
generated_part = file["DF_2303121152302944/O2genparticles"].arrays(library="pd")
good_generated_particles = generated_part[generated_part['fIndexMcCollisions'].isin(mccollision_pZ_cut['fIndexBCs'])]

print(len(good_generated_particles), len(generated_part))
good_generated_particles.head(10)

In [ ]:
# file by file...
all_files = file.keys(filter_name = r"DF_*")
file_ID = list(set([code.split("/")[0] for code in all_files]))
# remove the directoryes code
file_ID = [s for s in file_ID if not s.endswith(";1")]

good_gen_part = pd.DataFrame()
tot_gen_part = pd.DataFrame()

for directory in file_ID:
    
    # fPosZ cut
    mccollision_df = file[directory + "/O2mccollision"].arrays(library="pd")
    mccollision_pZ_cut = mccollision_df[mccollision_df["fPosZ"].abs() < 10]

    # "JOIN"
    generated_part = file[directory + "/O2genparticles"].arrays(library="pd")
    good_generated_particles = generated_part[generated_part['fIndexMcCollisions'].isin(mccollision_pZ_cut['fIndexBCs'])]

    #results
    good_gen_part = pd.concat([good_gen_part, good_generated_particles], ignore_index=True) 
    tot_gen_part = pd.concat([tot_gen_part, generated_part], ignore_index=True) 

print("The generated particles before check the Z coordinate are:", len(tot_gen_part))
print("After the cut on Z_collision < 10 cm we have ", len(good_gen_part), "particles")

In [ ]:
good_gen_part.head(3)

In [ ]:
# Select all the rows with fPdgCode = 421 
good_gen_part = good_gen_part[(good_gen_part["fPdgCode"]==421)]

#defined pT interval in which you can/want to “make the measurement”
# For now no selection on pT...TO DO LATER

# Pseudorapidity cut
good_gen_part = good_gen_part[good_gen_part["fMainMotherY"].abs() < 0.8]

# Check on the fMainBeautyAncestor = 0
good_gen_part = good_gen_part[good_gen_part["fMainBeautyAncestorY"].abs() == 0]

#Visualize the results
print("In the MC simulation have been generated", len(good_gen_part) ," D_0 mesons")

In [ ]:
n_tot_collisions = len(mccollision_pZ_cut_df)
print("Total number of collisions: ", n_tot_collisions)
n_fposZ = len(mccollision_pZ_cut)
print("Number of collisions with |fPosZ| < 10 cm: ", n_fposZ)

In [ ]:
# Select the particle from the O2filtertrackmc table
tracks_df = file["DF_2303121152302944/O2filtertrackmc"].arrays(library="pd")
selected_tracks = tracks_df[tracks_df["fMainMotherY"].abs() < 0.8]
selected_tracks.head()